# EDA - Store Sales Time Series Forecasting

Cilj: razumjeti podatke prije feature engineeringa i modeliranja.

Fajlovi:
- `train.csv` - dnevna prodaja po prodavnici i kategoriji (2013-2017)
- `stores.csv` - metadata prodavnica
- `transactions.csv` - broj transakcija po danu
- `oil.csv` - dnevna cijena nafte
- `holidays_events.csv` - praznici i eventi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)

DATA = '../data/raw/'

## 1. Ucitavanje podataka

In [ ]:
train        = pd.read_csv(DATA + 'train.csv', parse_dates=['date'])
stores       = pd.read_csv(DATA + 'stores.csv')
transactions = pd.read_csv(DATA + 'transactions.csv', parse_dates=['date'])
oil          = pd.read_csv(DATA + 'oil.csv', parse_dates=['date'])
holidays     = pd.read_csv(DATA + 'holidays_events.csv', parse_dates=['date'])

print(f'train:        {train.shape}')
print(f'stores:       {stores.shape}')
print(f'transactions: {transactions.shape}')
print(f'oil:          {oil.shape}')
print(f'holidays:     {holidays.shape}')

## 2. train.csv

In [ ]:
train.head(10)

In [ ]:
train.info()

In [ ]:
print('Missing values:')
print(train.isnull().sum())
print(f'\nPeriod: {train.date.min().date()} do {train.date.max().date()}')
print(f'Prodavnice: {train.store_nbr.nunique()}')
print(f'Kategorije: {train.family.nunique()}')
print(f'Redova sa sales=0: {(train.sales == 0).sum():,} ({(train.sales == 0).mean():.1%})')

In [ ]:
train.sales.describe()

In [ ]:
# Ukupna dnevna prodaja kroz cijeli period
daily = train.groupby('date')['sales'].sum()

fig, ax = plt.subplots()
ax.plot(daily.index, daily.values, linewidth=0.8, color='steelblue')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_title('Ukupna dnevna prodaja (sve prodavnice, sve kategorije)')
ax.set_ylabel('Sales')
plt.tight_layout()
plt.show()

In [ ]:
# Prodaja po kategoriji
by_family = train.groupby('family')['sales'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
by_family.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Ukupna prodaja po kategoriji proizvoda')
ax.set_xlabel('')
ax.set_ylabel('Sales')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Sezonalnost: prodaja po danu u sedmici
train['dayofweek'] = train.date.dt.day_name()
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
by_dow = train.groupby('dayofweek')['sales'].mean().reindex(dow_order)

fig, ax = plt.subplots(figsize=(9, 4))
by_dow.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Prosjecna prodaja po danu u sedmici')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Sezonalnost: prodaja po mjesecu
train['month'] = train.date.dt.month
by_month = train.groupby('month')['sales'].mean()

fig, ax = plt.subplots(figsize=(9, 4))
by_month.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Prosjecna prodaja po mjesecu')
ax.set_xlabel('Mjesec')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Efekat promocija
promo_effect = train.groupby('onpromotion')['sales'].mean()
print('Prosjecna prodaja - nula stavki na promociji:', round(train[train.onpromotion == 0]['sales'].mean(), 2))
print('Prosjecna prodaja - barem jedna stavka na promociji:', round(train[train.onpromotion > 0]['sales'].mean(), 2))

## 3. stores.csv

In [ ]:
stores.head()


In [ ]:
print('Tipovi prodavnica:', stores['type'].value_counts().to_dict())
print('Gradovi:', stores['city'].nunique())
print('Clusteri:', stores['cluster'].nunique())
stores.describe()

In [ ]:
# Prodaja po tipu prodavnice
train_stores = train.merge(stores, on='store_nbr')
by_type = train_stores.groupby('type')['sales'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
by_type.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Prosjecna prodaja po tipu prodavnice')
ax.set_xlabel('Tip')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. transactions.csv

In [ ]:
transactions.head()

In [ ]:
# Korelacija transakcija i prodaje
daily_sales = train.groupby('date')['sales'].sum().reset_index()
daily_trans = transactions.groupby('date')['transactions'].sum().reset_index()
merged = daily_sales.merge(daily_trans, on='date')

corr = merged['sales'].corr(merged['transactions'])
print(f'Korelacija (sales vs transactions): {corr:.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(merged['transactions'], merged['sales'], alpha=0.3, s=8, color='steelblue')
ax.set_title(f'Transakcije vs Prodaja (r={corr:.2f})')
ax.set_xlabel('Transakcije')
ax.set_ylabel('Sales')
plt.tight_layout()
plt.show()

## 5. oil.csv

In [ ]:
oil.head()


In [ ]:
print(f'Missing values: {oil.dcoilwtico.isnull().sum()} ({oil.dcoilwtico.isnull().mean():.1%})')
print(f'Period: {oil.date.min().date()} do {oil.date.max().date()}')

# Vikendi/praznici nemaju cijenu nafte - interpoliramo
oil_full = oil.set_index('date').reindex(
    pd.date_range(oil.date.min(), oil.date.max())
).interpolate().reset_index()
oil_full.columns = ['date', 'dcoilwtico']

fig, ax = plt.subplots()
ax.plot(oil_full.date, oil_full.dcoilwtico, linewidth=0.9, color='darkorange')
ax.set_title('Cijena nafte (WTI) - interpolovano za vikende')
ax.set_ylabel('USD/bbl')
plt.tight_layout()
plt.show()

In [ ]:
# Korelacija cijene nafte i prodaje
merged_oil = merged.merge(oil_full, on='date', how='left')
corr_oil = merged_oil['sales'].corr(merged_oil['dcoilwtico'])
print(f'Korelacija (sales vs cijena nafte): {corr_oil:.3f}')

## 6. holidays_events.csv

In [ ]:
holidays.head(10)

In [ ]:
print('Tipovi:', holidays['type'].value_counts().to_dict())
print('Locale:', holidays['locale'].value_counts().to_dict())
print('Transferred:', holidays['transferred'].value_counts().to_dict())

In [ ]:
# Efekat praznika na prodaju
holiday_dates = holidays[holidays['transferred'] == False]['date'].unique()

daily2 = train.groupby('date')['sales'].sum().reset_index()
daily2['is_holiday'] = daily2['date'].isin(holiday_dates)

effect = daily2.groupby('is_holiday')['sales'].mean()
print('Prosjecna dnevna prodaja:')
print(f'  Obican dan:  {effect[False]:,.0f}')
print(f'  Praznik:     {effect[True]:,.0f}')
print(f'  Razlika:     {(effect[True]/effect[False]-1):+.1%}')

## 7. Zakljucak i plan za feature engineering

**Sta smo naucili:**
- train ima X% nula - treba paziti na log transformaciju
- cijena nafte ima missing values za vikende - interpolacija
- jasna sedmicna i godisnja sezonalnost
- promocije znacajno uticu na prodaju
- praznici imaju efekt (gore/dole ovisno o tipu)

**Merge plan za master dataset:**
- train + stores (left join na `store_nbr`)
- + transactions (left join na `date` + `store_nbr`)
- + oil interpolovano (left join na `date`)
- + holidays agregirano po datumu (national) i gradu (local/regional)

**Features za modeliranje:**
- lag features (sales t-1, t-7, t-14, t-28)
- rolling mean (7d, 14d, 28d)
- kalendarske features (dayofweek, month, weekofyear)
- is_holiday, is_weekend, days_to_payday (15. i zadnji u mjesecu)
- cijena nafte (interpolovana)
- store tip, cluster
- onpromotion